# Service Alert Delay Correlation (Polars)

Compares observed delays during active service alerts with matched no-alert controls using the Polars analysis path.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

service_alerts = load_polars_script("polars_service_alert_delay_correlation", "service-alert-delay-correlation.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
GTFS_DIR = None
GTFS_ROOT = PROJECT_ROOT / "data" / "gtfs"
LIMIT = 20
MIN_OBSERVATIONS = 100
ALERT_KIND = "route"  # "route", "stop", or "any"
LINE_REF = None
START = None
END = None
ANALYSIS_DAYS = 2
FULL_HISTORY = False
TIMEZONE = "Europe/Helsinki"
QUALITY_MODE = "conservative"
BUCKET = "trip-stop"


Alert rows are grouped by cause, effect, priority, and route/stop scope. Controls are matched from non-alert buckets with the same line, direction, local hour, and weekday/weekend context.

Set `NO_CACHE = True` to read SQLite directly, or `FORCE_CACHE = True` to rebuild the Polars cache.

In [ ]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    force_cache = FORCE_CACHE
    no_cache = NO_CACHE
    gtfs_dir = GTFS_DIR
    gtfs_root = GTFS_ROOT
    limit = LIMIT
    min_observations = MIN_OBSERVATIONS
    alert_kind = ALERT_KIND
    line_ref = LINE_REF
    start = START
    end = END
    analysis_days = ANALYSIS_DAYS
    full_history = FULL_HISTORY
    timezone = TIMEZONE
    quality_mode = QUALITY_MODE
    exclude_stop_call_disagreement = False
    bucket = BUCKET

buckets = service_alerts.load_delay_buckets_for_args(Args)
if LINE_REF:
    buckets = buckets.filter(pl.col("line_ref") == LINE_REF)
window_start, window_end, window_description = service_alerts.resolve_analysis_window(Args, buckets)
if window_start is not None:
    buckets = buckets.filter(pl.col("representative_time_utc") >= window_start)
if window_end is not None:
    buckets = buckets.filter(pl.col("representative_time_utc") < window_end)
print(window_description)
grouped, line = service_alerts.build_alert_results(
    service_alerts.settings_from_args(Args),
    buckets,
    alert_kind=ALERT_KIND,
)
grouped


In [ ]:
line


In [ ]:
if line.is_empty():
    print("No matching observations found.")
else:
    fig = px.bar(
        line.sort("p90_delay_lift_min"),
        x="p90_delay_lift_min",
        y="line_ref",
        orientation="h",
        title="Delay lift during active service alerts by line",
        labels={"line_ref": "Line", "p90_delay_lift_min": "Alert minus no-alert P90 delay, minutes"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
